# Análise de Entropia padrão, condicional a classe e ganho
Este notebook calcula a entropia da classe e, do atributo, a entropia padrão, condicional a classe e ganho.

In [1]:
import pandas as pd
from scipy.stats import entropy

In [2]:
base = pd.read_csv("database-5.csv")
classe = "Q092"

def entropia_padrao(column):
    freqrel = column.value_counts(normalize=True)
    return entropy(freqrel, base=2)

def entropia_condicional(atributo, classe):
    df = pd.concat([atributo, classe], axis=1)
    valores = atributo.unique()
    total = len(atributo)
    entropia_cond = 0.0
    for valor in valores:
        subset = df[df[atributo.name] == valor][classe.name]
        proporcao = len(subset) / total
        entropia_subset = entropia_padrao(subset)
        entropia_cond += proporcao * entropia_subset
    return entropia_cond

In [3]:
entropia_classe = entropia_padrao(base[classe])
print(f"Entropia da classe: {round(entropia_classe,4)}")
print("-" * 95)

entropias_cond = {}
ganhos = {}

print(f"{'Atributo':<30} {'Entropia Padrão':>18} {'Entropia Condicional':>22} {'Ganho de Informação':>22}")
print("-" * 95)

for atributo in base.columns:
    if atributo == classe:
        continue
    entropia_atr = entropia_padrao(base[atributo])
    entropia_cond = entropia_condicional(base[atributo], base[classe])
    entropias_cond[atributo] = entropia_cond
    ganho = entropia_classe - entropia_cond
    ganhos[atributo] = ganho
    print(f"{atributo:<30} {entropia_atr:>18.4f} {entropia_cond:>22.4f} {ganhos[atributo]:>22.4f}")


Entropia da classe: 0.5012
-----------------------------------------------------------------------------------------------
Atributo                          Entropia Padrão   Entropia Condicional    Ganho de Informação
-----------------------------------------------------------------------------------------------
A001                                       0.7053                 0.5008                 0.0004
A002010                                    0.0312                 0.5012                 0.0000
A005012                                    0.5270                 0.5010                 0.0001
A01501                                     0.6335                 0.5012                 0.0000
C004                                       0.8905                 0.5012                 0.0000
C006                                       0.9980                 0.4791                 0.0220
C011                                       1.4689                 0.4972                 0.0040
C014         

In [4]:
ganhos_ordenados = sorted(ganhos.items(), key=lambda x: x[1], reverse=True)

print("\nAtributos ordenados por ganho de informação:")
print("-" * 51)
print(f"{'Atributo':<30} {'Ganho de Informação':>20}")
print("-" * 51)

for atributo, ganho in ganhos_ordenados:
    print(f"{atributo:<30} {ganho:>20.4f}")



Atributos ordenados por ganho de informação:
---------------------------------------------------
Atributo                        Ganho de Informação
---------------------------------------------------
N016                                         0.0733
N010                                         0.0528
N012                                         0.0496
N017                                         0.0489
N013                                         0.0443
N011                                         0.0393
N015                                         0.0376
N00101                                       0.0348
N014                                         0.0342
N018                                         0.0239
C006                                         0.0220
Violencia_Psicologica                        0.0144
Violencia_Sexual                             0.0137
E01401                                       0.0123
Carga_Horaria_Semanal                        0.0099
Salario           

In [5]:
frequencia_minima = 0.05
ganho_minimo = 0.01

atributos_a_remover = []
atributos_a_preservar = []

In [6]:
for atributo in base.columns:
    if atributo in ["Q092", "D00901", "C004", "Tempo_Minimo_Exercicio", "TempoDeTela", "Frequencia_Fumo"]:
        continue
    proporcoes = base[atributo].value_counts(normalize=True)
    if (proporcoes < frequencia_minima).any():
        ganho = ganhos[atributo]
        if ganho < ganho_minimo:
            atributos_a_remover.append(atributo) 
        else:
            atributos_a_preservar.append((atributo, ganho)) 

base_filtrada = base.drop(columns=atributos_a_remover)

In [7]:
print("Atributos preservados (baixa frequência, mas ganho ≥ 0.05):")
for nome, ganho in atributos_a_preservar:
    print(f"  {nome:<30} Ganho: {ganho:.4f}")

print("\nAtributos removidos (baixa frequência e ganho < 0.01):")
for nome in atributos_a_remover:
    print(f"  {nome:<30} Ganho: {ganhos[nome]:.4f}")

print(f"\nTotal de atributos restantes na base: {base_filtrada.shape[1]}")

Atributos preservados (baixa frequência, mas ganho ≥ 0.05):
  E01401                         Ganho: 0.0123
  N00101                         Ganho: 0.0348
  N012                           Ganho: 0.0496
  N013                           Ganho: 0.0443
  N014                           Ganho: 0.0342
  N015                           Ganho: 0.0376
  N016                           Ganho: 0.0733
  N017                           Ganho: 0.0489
  N018                           Ganho: 0.0239
  Violencia_Psicologica          Ganho: 0.0144

Atributos removidos (baixa frequência e ganho < 0.01):
  A001                           Ganho: 0.0004
  A002010                        Ganho: 0.0000
  C011                           Ganho: 0.0040
  D001                           Ganho: 0.0000
  E011                           Ganho: 0.0066
  E033                           Ganho: 0.0046
  M01701                         Ganho: 0.0003
  P03301                         Ganho: 0.0016
  P03302                         Ganho

In [8]:
corte_entropia_condicional = 0.5

atributos_alta_entropia = [
    atributo for atributo, entropia in entropias_cond.items()
    if atributo not in ["D00901", "C004", "Tempo_Minimo_Exercicio", "TempoDeTela", "Frequencia_Fumo"]
        if entropia > corte_entropia_condicional and atributo in base_filtrada.columns]
base_filtrada2 = base_filtrada.drop(columns=atributos_alta_entropia)

print("Atributos removidos por alta entropia condicional (> 0.5):")
for atributo in atributos_alta_entropia:
    print(f"  {atributo:<30} Entropia condicional: {entropias_cond[atributo]:.4f}")

print(f"\nTotal de atributos restantes após filtro por entropia condicional: {base_filtrada2.shape[1]-1}")


Atributos removidos por alta entropia condicional (> 0.5):
  A005012                        Entropia condicional: 0.5010
  A01501                         Entropia condicional: 0.5012
  D00201                         Entropia condicional: 0.5011
  M005010                        Entropia condicional: 0.5012
  M009                           Entropia condicional: 0.5012
  M011021                        Entropia condicional: 0.5008
  M011061                        Entropia condicional: 0.5012
  M01501                         Entropia condicional: 0.5001
  M01601                         Entropia condicional: 0.5002
  M01901                         Entropia condicional: 0.5003
  P03001                         Entropia condicional: 0.5008
  Pratica_Exercicio              Entropia condicional: 0.5010

Total de atributos restantes após filtro por entropia condicional: 29


In [9]:
for atributo in base_filtrada2.columns:
    if atributo == classe:
        continue
    entropia_atr = entropia_padrao(base[atributo])
    entropia_cond = entropia_condicional(base[atributo], base[classe])
    ganho = entropia_classe - entropia_cond
    print(f"{atributo:<30} {entropia_atr:>18.4f} {entropia_cond:>22.4f} {ganhos[atributo]:>22.4f}")

C004                                       0.8905                 0.5012                 0.0000
C006                                       0.9980                 0.4791                 0.0220
C014                                       0.9290                 0.4988                 0.0024
D00901                                     1.6610                 0.5004                 0.0008
E01401                                     2.3286                 0.4889                 0.0123
E01403                                     0.9843                 0.4979                 0.0033
J05301                                     0.3124                 0.4959                 0.0052
M01401                                     1.1329                 0.4989                 0.0023
N00101                                     0.8337                 0.4664                 0.0348
N010                                       1.4284                 0.4484                 0.0528
N011                                    

In [10]:
base_filtrada2.to_csv("base_tratada.csv", index=False)